# CHECKPOINT 4.2: QUẢN LÝ LỚP HỌC & ENROLLMENT (MIDDLE SCHOOL ERP)

Notebook này phân tích và kiểm thử toàn diện các quy tắc nghiệp vụ quản lý Lớp học, Enrollment và Chuyển lớp trong hệ thống **SmartEdu**:
1. **Mô hình Khối & Lớp:** Khối 6, 7, 8, 9 với 12 lớp chuẩn, sĩ số 18 học sinh/lớp.
2. **Single Source of Truth (`classEnrollments`):** Quản trị lịch sử học tập, không phụ thuộc cache.
3. **Quy tắc 1 Active Enrollment:** Trong cùng một năm học, mỗi học sinh chỉ có tối đa 1 enrollment ACTIVE.
4. **Chuyển lớp nguyên tử (Atomic Transfer):** Thực thi qua Firestore Batch để đảm bảo ACID và bảo toàn lịch sử không xóa cứng.
5. **Kiểm soát dung lượng lớp:** Sức chứa không vượt quá 50 và không được giảm thấp hơn sĩ số đang học.

In [ ]:
import json
from datetime import datetime

VALID_GRADES = [6, 7, 8, 9]
DEFAULT_CAPACITY = 18
DEFAULT_ACADEMIC_YEAR = '2026-2027'

print(f"Quy chuẩn Trung tâm: Khối {VALID_GRADES}, Sĩ số {DEFAULT_CAPACITY} HS/lớp, Năm học: {DEFAULT_ACADEMIC_YEAR}")

## 1. Kiểm thử Logic Tính Sĩ số Thực tế & Ràng buộc Khối Lớp

In [ ]:
def calculate_class_size(class_id, academic_year, enrollments):
    return len([
        e for e in enrollments 
        if e['classId'] == class_id and 
           e['academicYear'] == academic_year and 
           e['status'] in ['ACTIVE', 'Đang học']
    ])

def validate_grade(grade):
    if grade not in VALID_GRADES:
        return False, f"Khối {grade} không hợp lệ. Chỉ chấp nhận các khối 6, 7, 8, 9."
    return True, "Hợp lệ"

# Kiểm thử
assert validate_grade(6)[0] == True
assert validate_grade(9)[0] == True
assert validate_grade(10)[0] == False
assert validate_grade(5)[0] == False
print("✓ PASS: Ràng buộc Khối lớp 6-9 chính xác!")

## 2. Mô phỏng Quy trình Chuyển Lớp Nguyên Tử (Atomic Transfer)

In [ ]:
def execute_transfer(student, from_class, to_class, reason, enrollments):
    # 1. Validation
    if from_class['id'] == to_class['id']:
        raise ValueError("Lớp chuyển đến phải khác lớp hiện tại.")
    if student['grade'] != to_class['grade']:
        raise ValueError(f"Không thể chuyển học sinh khối {student['grade']} sang lớp khối {to_class['grade']}.")
    
    target_size = calculate_class_size(to_class['id'], to_class['academicYear'], enrollments)
    if target_size >= to_class['capacity']:
        raise ValueError(f"Lớp {to_class['name']} đã đủ sĩ số ({target_size}/{to_class['capacity']}).")
    
    now = datetime.now().isoformat()
    
    # 2. Cập nhật bản ghi enrollment cũ
    old_enrollment = next(e for e in enrollments if e['studentId'] == student['id'] and e['classId'] == from_class['id'] and e['status'] == 'ACTIVE')
    old_enrollment['status'] = 'TRANSFERRED'
    old_enrollment['endDate'] = now
    old_enrollment['reason'] = reason
    old_enrollment['toClassId'] = to_class['id']
    
    # 3. Tạo bản ghi enrollment mới
    new_enrollment = {
        'id': f"ENR-{len(enrollments)+1:03d}",
        'studentId': student['id'],
        'studentName': student['name'],
        'classId': to_class['id'],
        'className': to_class['name'],
        'grade': student['grade'],
        'academicYear': to_class['academicYear'],
        'startDate': now,
        'status': 'ACTIVE',
        'fromClassId': from_class['id'],
        'reason': reason
    }
    enrollments.append(new_enrollment)
    
    # 4. Cập nhật cache của học sinh
    student['classId'] = to_class['id']
    student['className'] = to_class['name']
    
    return student, enrollments

# Kiểm thử mô phỏng
mock_student = {'id': 'STU-001', 'name': 'Nguyễn Minh Anh', 'grade': 6, 'classId': 'class_6A1', 'className': 'Lớp 6A1'}
cls_6A1 = {'id': 'class_6A1', 'name': 'Lớp 6A1', 'grade': 6, 'capacity': 18, 'academicYear': '2026-2027'}
cls_6A2 = {'id': 'class_6A2', 'name': 'Lớp 6A2', 'grade': 6, 'capacity': 18, 'academicYear': '2026-2027'}
mock_enrollments = [{'id': 'ENR-001', 'studentId': 'STU-001', 'classId': 'class_6A1', 'className': 'Lớp 6A1', 'academicYear': '2026-2027', 'status': 'ACTIVE'}]

updated_stu, next_enrolls = execute_transfer(mock_student, cls_6A1, cls_6A2, 'Cân bằng sĩ số', mock_enrollments)

assert updated_stu['classId'] == 'class_6A2'
assert len(next_enrolls) == 2
assert next_enrolls[0]['status'] == 'TRANSFERRED'
assert next_enrolls[1]['status'] == 'ACTIVE'
print("✓ PASS: Nghiệp vụ chuyển lớp nguyên tử (Atomic Transfer) bảo toàn toàn bộ lịch sử!")